In [1]:
import numpy as np
from collections import Counter
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# --- 1. Prepare the Dataset ---
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the quick brown dog jumps over the lazy fox",
    "a man is walking on the street",
    "a dog is barking at the cat",
    "the street is busy with people",
]

# --- 2. Tokenize and Create Word-to-Index Mappings ---
# Index 0 is reserved for padding; words start at 1 (most frequent first).
tokenized = [sentence.lower().split() for sentence in corpus]
counts = Counter(word for sent in tokenized for word in sent)
word_index = {word: i + 1 for i, (word, _) in enumerate(counts.most_common())}
vocab_size = len(word_index) + 1
print(f"Vocabulary Size: {vocab_size}")

sequences = [[word_index[w] for w in sent] for sent in tokenized]

# --- 3. Generate CBOW Training Data ---
window_size = 2
data = []

for sequence in sequences:
    for i, word in enumerate(sequence):
        context = []
        for j in range(i - window_size, i + window_size + 1):
            if j != i and 0 <= j < len(sequence):
                context.append(sequence[j])
        if context:
            data.append((context, word))

X_context = [item[0] for item in data]
y_target = np.array([item[1] for item in data])

# Pad contexts with 0 (post-padding) to a uniform length
max_len = max(len(x) for x in X_context)
X_context_padded = np.zeros((len(X_context), max_len), dtype="int32")
for i, ctx in enumerate(X_context):
    X_context_padded[i, : len(ctx)] = ctx

y_target_one_hot = to_categorical(y_target, num_classes=vocab_size)

# --- 4. Build the CBOW Model (Functional API) ---
embedding_dim = 100

input_context = Input(shape=(max_len,), name="Context_Input")

# mask_zero=True makes the model ignore the padding (index 0)
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="Word_Embeddings",
)(input_context)

# Averages only the real (unmasked) context words
avg_context = GlobalAveragePooling1D(name="Average_Context")(embedding_layer)

output_layer = Dense(
    vocab_size, activation="softmax", name="Target_Prediction"
)(avg_context)

cbow_model = Model(inputs=input_context, outputs=output_layer)

# --- 5. Compile and Train the Model ---
cbow_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
cbow_model.summary()

print("\nTraining the CBOW model...")
cbow_model.fit(X_context_padded, y_target_one_hot, epochs=500, verbose=0)
print("Training complete.")

# --- 6. Extract Word Embeddings ---
embedding_matrix = cbow_model.get_layer("Word_Embeddings").get_weights()[0]

print("\nLearned Word Embeddings (First 5 Dimensions shown):")
for word, idx in list(word_index.items())[:10]:
    print(f"Word: '{word:<10}' Vector: {embedding_matrix[idx][:5]}")

Vocabulary Size: 21


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Context_Input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Word_Embeddings     │ (None, 4, 100)    │      2,100 │ Context_Input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 4)         │          0 │ Context_Input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Average_Context     │ (None, 100)       │          0 │ Word_Embeddings[… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Target_Prediction   │ (None, 21)        │      2,121 │ Average_Context[… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,221 (16.49 KB)

 Trainable params: 4,221 (16.49 KB)

 Non-trainable params: 0 (0.00 B)


Training the CBOW model...
Training complete.

Learned Word Embeddings (First 5 Dimensions shown):
Word: 'the       ' Vector: [ 0.31263798  0.5343465   0.5619984  -0.16313775  0.67437726]
Word: 'dog       ' Vector: [ 0.5276418  -0.34813395  0.4054321  -0.4955442   0.16372377]
Word: 'is        ' Vector: [-0.82611805 -0.23866637  0.02820979 -0.18089953 -0.00680186]
Word: 'quick     ' Vector: [ 0.32668763  0.1174102  -0.10860444  0.11578023 -0.22552668]
Word: 'brown     ' Vector: [ 0.29952705 -0.3484444  -0.05960737  1.0164663  -0.18517497]
Word: 'fox       ' Vector: [ 0.41044807 -0.32454062  0.36881027 -0.36106846 -0.01767098]
Word: 'jumps     ' Vector: [ 0.34624493  0.28309706  0.27329835 -0.23689869  0.25370678]
Word: 'over      ' Vector: [ 0.5022224   0.37648728 -0.03682803  0.22845717 -0.617966  ]
Word: 'lazy      ' Vector: [-0.3326919  -0.22192845 -0.44859728  0.3381836  -0.2260523 ]
Word: 'a         ' Vector: [ 0.30719978 -0.0765931  -0.35322678 -0.12250156  0.2875807 ]
